# Verify Table `tab:spatial` — DARTS on pavia4 (bitumen, theta=.15, 5 seeds)
Runs the `rebuttal_shir` pipeline itself (published noise-AFTER training in
whitened space, published scoring, training-quantile thresholds, per-class
false-alarm metrics) with DARTS only. The CONFIG cell exposes the front
(`'zca'` = the exact table configuration with eig-floor whitening;
`'std'` = per-band std), the full DARTS architecture and training config,
lambda for the CFAR-normalized statistic, and the published RNG protocol
(DART trained first, DARTS inheriting the stream — as the paper's pavia4
run did). Reports per-seed and mean pAUC / AUC / Pd@0.05 / Pfa-max for
BOTH the plain and the lambda-normalized DARTS statistic, next to the
table's target row. Saves checkpoints + score maps; restart-safe.

In [ ]:
%cd /content
!rm -rf repo final-paper-experiment-rebuttal_shir rebuttal_shir.tar.gz
!wget -q --show-progress https://github.com/michaelpiro/final-paper-experiment/archive/refs/heads/rebuttal_shir.tar.gz
!tar xzf rebuttal_shir.tar.gz --exclude='*/repro/checkpoints/*'
!mv final-paper-experiment-rebuttal_shir repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/pavia-u.mat'), 'missing data'

In [ ]:
# ======================= CONFIG — edit me =======================
CONFIG = {
  'theta': 0.15,
  'tag': '',                 # config label -> part of the run key (bump when
                             # changing arch/training so old rows don't collide)
  'seeds': [42, 43, 44, 45, 46],
  'front': 'std',            # 'zca' = exact table config | 'std' = per-band std
  'lam': 0.1,                # CFAR local-moment lambda (the table's lambda)
  'ablation_lams': [0.0, 0.1, 0.9, 1.0],   # lambda-sweep cell
  'published_rng': True,     # train DART first; DARTS inherits the RNG stream
                             # (the published pavia4 protocol)
  # ---- DARTS model + training (rebuttal_shir spatial.yaml defaults) ----
  'darts': {
    'd_lat': 16, 'K': 7, 'enc_hidden': [64, 32], 'score_hidden': [512],
    'activation': 'relu',
    'lr': 3e-4, 'batch_size': 512, 'weight_decay': 1e-5,
    'epochs': 1000,
    'whiten_eig_floor': 1e-5,   # used by the 'zca' front
    'dsm_sigma_rho': 0.1,       # sigma = sqrt(rho), noise in WHITENED space
  },
  # table row being verified (DARTS, lambda=.1):
  'target_row': {'pauc': 0.556, 'auc': 0.926, 'pd05': 0.730,
                 'pfa_max': 0.088},
}
# ================================================================
import json
json.dump(CONFIG, open('verify_config.json', 'w'), indent=1)
print(json.dumps(CONFIG, indent=1))

In [ ]:
%%writefile run_verify.py
"""DARTS-only verification of the spatial table on pavia4.
Published training loop (noise-after in whitened space), published scoring
and metrics (_row: pAUC/AUC/Pd05/per-class Pfa, thresholds = alpha-quantile
of training scores). Front switchable: zca (table) | std."""
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
from tqdm import tqdm

from repro import scenes
from repro.protocols.spatial import load_cfg, _row, _windows
from repro.core.metrics import cfar_threshold
from repro.core.data import Whitening, plant_targets
from repro.core.seeding import seed_all
from repro.models.dart import DART
from repro.models.darts.model import DARTS, _NeighborDenoiser

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
CFG = json.load(open('verify_config.json'))
DCFG = CFG['darts']
THETA = float(CFG['theta'])
LAM = float(CFG['lam'])
FRONT = CFG['front']
OUT_JSON = 'results_verify.json'
CKPT_ROOT = 'ckpt_verify'

SP_CFG = load_cfg()
os.makedirs(CKPT_ROOT, exist_ok=True)


def make_front(tr):
    if FRONT == 'zca':
        return Whitening.from_data(
            np.asarray(tr, np.float32),
            eig_floor=float(DCFG['whiten_eig_floor'])).to(DEVICE)
    X = np.asarray(tr, np.float64)
    return Whitening(X.mean(0).astype(np.float32),
                     np.diag(1.0 / X.std(0)).astype(np.float32)).to(DEVICE)


def fit_darts(tr_raw, tr_nbr, seed, reseed):
    """Verbatim published DARTS.fit, except the front comes from make_front."""
    D = tr_raw.shape[1]
    if reseed:
        seed_all(seed)
    Wh = make_front(tr_raw)
    sigma = float(np.sqrt(DCFG['dsm_sigma_rho']))
    net = _NeighborDenoiser(D, int(DCFG['d_lat']), int(DCFG['K']),
                            list(DCFG['enc_hidden']),
                            list(DCFG['score_hidden']), sigma,
                            DCFG['activation'], Wh).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=float(DCFG['lr']),
                            weight_decay=float(DCFG['weight_decay']))
    X = torch.tensor(np.asarray(tr_raw, np.float32), device=DEVICE)
    N = torch.tensor(np.asarray(tr_nbr, np.float32), device=DEVICE)
    P, B, E = len(X), int(DCFG['batch_size']), int(DCFG['epochs'])
    best_loss, best_state = float('inf'), None
    pbar = tqdm(range(E), desc=f'DARTS s{seed}', dynamic_ncols=True,
                ascii=True, mininterval=5.0)
    for ep in pbar:
        perm = torch.randperm(P, device=DEVICE)
        tot, nb = 0.0, 0
        for i in range(0, P, B):
            sel = perm[i:i + B]
            x_w = net.whitening(X[sel])
            nbr_w = net.whitening(N[sel])
            eps = torch.randn_like(x_w) * sigma
            target = -eps / (sigma ** 2)
            score = net._forward_inner(x_w + eps, nbr_w)
            loss = ((score - target) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss.item()); nb += 1
        last = tot / max(nb, 1)
        pbar.set_postfix(loss=f'{last:.4f}')
        if last < best_loss:
            import copy as _copy
            best_loss = last
            best_state = _copy.deepcopy(net.state_dict())
    if best_state is not None:
        net.load_state_dict(best_state)
    det = DARTS(DCFG)
    det.net = net.eval()
    return det


def main():
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    scene = scenes.build('pavia4', SP_CFG)
    k = int(SP_CFG['k'])
    _, tr_nbr = _windows(scene['tr'], scene['tr_shape'], k, DEVICE)
    _, te_nbr = _windows(scene['te'], scene['te_shape'], k, DEVICE)
    r0, r1, c0, c1 = scene['test_box']
    te_gt = np.asarray(scene['gt'], int)[r0:r1, c0:c1].ravel()
    alpha = float(SP_CFG['alpha'])
    sig = np.asarray(scene['sig'], np.float32)
    for seed in [int(s) for s in CFG['seeds']]:
        tag = ('-' + CFG['tag']) if CFG.get('tag') else ''
        key = f'darts_{FRONT}{tag}_s{seed}'
        if key in res:
            print('skip (done):', key); continue
        t0 = time.time()
        if CFG.get('published_rng'):
            seed_all(seed)
            dart_cfg = dict(SP_CFG['dart'])
            w = (SP_CFG.get('net_width') or {}).get('pavia4')
            if w:
                dart_cfg['hidden'] = [int(w)]
            DART(dart_cfg).fit(scene['tr'], seed, DEVICE,
                               ckpt=os.path.join(CKPT_ROOT,
                                                 f'dart_s{seed}.pt'))
            reseed = False
        else:
            reseed = True
        if FRONT == 'zca':
            # verbatim published class fit (builds its own ZCA front)
            det = DARTS(DCFG).fit(scene['tr'], tr_nbr, seed, DEVICE,
                                  ckpt=None, reseed=reseed)
        else:
            det = fit_darts(scene['tr'], tr_nbr, seed, reseed=reseed)
        torch.save({'net': det.net.state_dict()},
                   os.path.join(CKPT_ROOT, f'{key}.pt'))
        tr_plain = det.score(scene['tr'], tr_nbr, scene['tr'], tr_nbr, sig)
        tr_cfar = DARTS.local_moment_normalize(
            tr_plain, scene['tr_shape'],
            win=int(SP_CFG.get('darts_cfar_window') or k),
            guard=int(SP_CFG.get('darts_cfar_guard', 1)), cfar_lam=LAM)
        thr = {'plain': cfar_threshold(np.asarray(tr_plain, float),
                                       target_fpr=alpha),
               'cfar': cfar_threshold(np.asarray(tr_cfar, float),
                                      target_fpr=alpha)}
        planted, labels, _ = plant_targets(
            scene['te'], sig, THETA, float(SP_CFG['target_fraction']),
            model='additive', seed=seed, spatial_shape=scene['te_shape'],
            edge_guard=int(SP_CFG['edge_guard']))
        planted = planted.astype(np.float32)
        sc_plain = det.score(planted, te_nbr, scene['tr'], tr_nbr, sig)
        sc_cfar = DARTS.local_moment_normalize(
            sc_plain, scene['te_shape'],
            win=int(SP_CFG.get('darts_cfar_window') or k),
            guard=int(SP_CFG.get('darts_cfar_guard', 1)), cfar_lam=LAM)
        np.savez_compressed(
            os.path.join(CKPT_ROOT, f'scores_{key}.npz'),
            plain=sc_plain, cfar=sc_cfar, labels=np.asarray(labels, np.int8))
        rows = {}
        for name, sc_, th in (('plain', sc_plain, thr['plain']),
                              ('cfar', sc_cfar, thr['cfar'])):
            rows[name] = _row(labels, sc_, th, alpha, te_gt)
            print(f"[s{seed} {name:<5}] pauc={rows[name]['pauc']:.3f} "
                  f"auc={rows[name]['auc']:.3f} "
                  f"pd05={rows[name]['pd05']:.3f} "
                  f"pfa_max={rows[name].get('pfa_max', float('nan')):.3f}",
                  flush=True)
        res[key] = {'seed': seed, 'front': FRONT, 'lam': LAM,
                    'rows': rows, 'cfg': DCFG,
                    'sec': round(time.time() - t0)}
        json.dump(res, open(OUT_JSON, 'w'), indent=1)
    print('ALL DONE', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
# ---- run the verification (5 seeds) ----
!python run_verify.py

In [ ]:
# ---- Summary vs the table row ----
import json
import numpy as np
CFG = json.load(open('verify_config.json'))
res = json.load(open('results_verify.json'))
tgt = CFG['target_row']
for stat in ('cfar', 'plain'):
    rows = [v['rows'][stat] for v in res.values()
            if v['front'] == CFG['front']]
    if not rows: continue
    print(f"\n== DARTS ({CFG['front']} front, {stat} statistic, "
          f"{len(rows)} seeds) ==")
    print(f"{'metric':<9} {'mean':>7} {'std':>6} {'table':>7}")
    for m, tk in (('pauc', 'pauc'), ('auc', 'auc'), ('pd05', 'pd05'),
                  ('pfa_max', 'pfa_max')):
        vals = [r.get(m, float('nan')) for r in rows]
        print(f'{m:<9} {np.mean(vals):>7.3f} {np.std(vals):>6.3f} '
              f'{tgt[tk]:>7.3f}')
print('\n(table row = the lambda-normalized DARTS statistic; '
      "for the exact table config set front='zca')")

In [ ]:
# ---- lambda ablation rows (saved checkpoints; no retraining) ----
import numpy as np, torch
from run_verify import (CFG, DCFG, FRONT, SP_CFG, CKPT_ROOT, make_front,
                        _NeighborDenoiser, DARTS, cfar_threshold, _row,
                        _windows, DEVICE)
from repro import scenes

LAMS = CFG.get('ablation_lams', [0.0, 0.1, 0.9, 1.0])
tag = ('-' + CFG['tag']) if CFG.get('tag') else ''
scene = scenes.build('pavia4', SP_CFG)
kwin = int(SP_CFG.get('darts_cfar_window') or SP_CFG['k'])
guard = int(SP_CFG.get('darts_cfar_guard', 1))
alpha = float(SP_CFG['alpha'])
_, tr_nbr = _windows(scene['tr'], scene['tr_shape'], int(SP_CFG['k']), DEVICE)
r0, r1, c0, c1 = scene['test_box']
te_gt = np.asarray(scene['gt'], int)[r0:r1, c0:c1].ravel()
sig = np.asarray(scene['sig'], np.float32)

acc = {lam: [] for lam in LAMS}
for seed in [int(s) for s in CFG['seeds']]:
    key = f'darts_{FRONT}{tag}_s{seed}'
    blob = torch.load(f'{CKPT_ROOT}/{key}.pt', map_location=DEVICE)
    net = _NeighborDenoiser(scene['tr'].shape[1], int(DCFG['d_lat']),
                            int(DCFG['K']), list(DCFG['enc_hidden']),
                            list(DCFG['score_hidden']),
                            float(np.sqrt(DCFG['dsm_sigma_rho'])),
                            DCFG['activation'],
                            make_front(scene['tr'])).to(DEVICE)
    net.load_state_dict(blob['net'])
    det = DARTS(DCFG); det.net = net.eval()
    tr_plain = det.score(scene['tr'], tr_nbr, scene['tr'], tr_nbr, sig)
    z = np.load(f'{CKPT_ROOT}/scores_{key}.npz')
    for lam in LAMS:
        tr_n = DARTS.local_moment_normalize(tr_plain, scene['tr_shape'],
                                            win=kwin, guard=guard,
                                            cfar_lam=lam)
        te_n = DARTS.local_moment_normalize(z['plain'], scene['te_shape'],
                                            win=kwin, guard=guard,
                                            cfar_lam=lam)
        thr = cfar_threshold(np.asarray(tr_n, float), target_fpr=alpha)
        acc[lam].append(_row(z['labels'], te_n, thr, alpha, te_gt))
    print(f'{key}: done')
print(f"\nlambda-sweep, {FRONT} front, mean over "
      f"{len(CFG['seeds'])} seeds (LaTeX-ready):")
for lam in LAMS:
    m = {k_: float(np.mean([r[k_] for r in acc[lam]]))
         for k_ in ('pauc', 'auc', 'pd05', 'pfa_max')}
    print(f"Ablation: DARTS & {lam} & {m['pauc']:.3f} & {m['auc']:.3f} & "
          f"{m['pd05']:.3f} & {m['pfa_max']:.3f}\\\\")

In [ ]:
# ---- Archive ----
import shutil, os
!zip -q -r verify_results.zip results_verify.json verify_config.json ckpt_verify
print(os.path.getsize('verify_results.zip')/1e6, 'MB')
from google.colab import files
shutil.copy('verify_results.zip', 'verify_results_dl.zip')
files.download('verify_results_dl.zip')